get_match_dict(match_id) — to get the raw match metadata
scrape_player_match_stats(match_id) — likely your main source for:
defense contributions
tackles won
interceptions
last man tackles
clearances

get_match_id_from_url(match_url) — if you start from Sofascore URLs
get_team_names(match_id) — useful for labeling
get_match_player_ids(match_id) — helpful if you need player-level joins

In [8]:
# scrape_sofascore.py

from ScraperFC.sofascore import Sofascore
import pandas as pd

sf = Sofascore()

MATCH_IDS = [
    # replace with real match IDs or Sofascore URLs
    # 12345678,
]

def get_match_metadata(match_id):
    """Extract basic match info from Sofascore match dict."""
    match = sf.get_match_dict(match_id)

    # You may need to adjust these keys after inspecting the dict once
    metadata = {
        "match_id": match.get("id"),
        "home_team": match.get("homeTeam", {}).get("name"),
        "away_team": match.get("awayTeam", {}).get("name"),
        "home_score": match.get("homeScore", {}).get("current"),
        "away_score": match.get("awayScore", {}).get("current"),
        "tournament": match.get("tournament", {}).get("name"),
        "season": match.get("season", {}).get("name"),
        "status": match.get("status", {}).get("description"),
        "start_time": match.get("startTimestamp"),
    }

    return metadata

def standardize_player_stats(df, match_id):
    """Keep only the columns we care about and add match_id."""
    df = df.copy()
    df["match_id"] = match_id

    # Print columns once so you can map names
    print("\nPlayer stats columns:")
    print(df.columns.tolist())

    return df

all_rows = []

for match_id in MATCH_IDS:
    try:
        print(f"Scraping match: {match_id}")

        metadata = get_match_metadata(match_id)
        player_stats = sf.scrape_player_match_stats(match_id)

        if player_stats is None or player_stats.empty:
            print(f"No player stats found for match {match_id}")
            continue

        player_stats = standardize_player_stats(player_stats, match_id)

        # add metadata to every player row
        for key, value in metadata.items():
            player_stats[key] = value

        all_rows.append(player_stats)

    except Exception as e:
        print(f"Failed for match {match_id}: {e}")

if all_rows:
    final_df = pd.concat(all_rows, ignore_index=True)
    final_df.to_csv("sofascore_player_match_stats.csv", index=False)
    print("\nSaved to sofascore_player_match_stats.csv")
else:
    print("\nNo data collected.")


No data collected.


In [10]:
match_data = get_match_metadata(12813003) 

In [ ]:
match_data

AttributeError: 'dict' object has no attribute 'head'